In [46]:
from typing import Tuple, Optional, Union
import os
import numpy as np

from PIL import Image
from pain2map import PainMode

In [48]:
BASE_PATH = os.path.join("D:\\", "Workspaces", "vscode-workspace", "ai_x_medicine", "data")
WORLD_PATH = os.path.join(BASE_PATH, "countries_map.zip")
OUTPUT_PATH = os.path.join(BASE_PATH, "out", "maps_new")
RESIZED_PATH = os.path.join(BASE_PATH, "out", "maps_resized")

for dir_path in [OUTPUT_PATH, RESIZED_PATH]:
    if not os.path.isdir(dir_path):
        # Create the directory
        try:
            os.mkdir(OUTPUT_PATH)
            print(f"Directory '{OUTPUT_PATH}' created successfully.")
        except FileExistsError:
            print(f"Directory '{OUTPUT_PATH}' already exists.")
        except PermissionError:
            print(f"Permission denied: Unable to create '{OUTPUT_PATH}'.")
        except Exception as e:
            print(f"An error occurred: {e}")

In [47]:
MAP_SIZE_COUNTRIES = (2000, 1000)
MAP_SIZE_SOUNDS = (600, 300)
MAP_SIZE_CRATERS = (1000, 500)
MAP_SIZE_DEFAULT = (2000, 1000)

Image.MAX_IMAGE_PIXELS = 20_000 * 10_000

In [49]:
# Util functions
def get_abbr(mode: Union[PainMode, str]) -> str:
    if isinstance(mode, PainMode):
        return mode.abbreviation
    return mode

def get_map_path(mode: Union[PainMode, str]) -> str:
    return os.path.join(OUTPUT_PATH, f"{get_abbr(mode)}.png")

def get_filename_from_size(mode: Union[PainMode, str], target_size: Optional[Tuple[int, int]] = None) -> str:
    abbr = get_abbr(mode)
    if target_size is None:
        return abbr
    return f"{abbr}_{target_size[0]}x{target_size[1]}"

def get_resized_map_path(mode: Union[PainMode, str], target_size: Optional[Tuple[int, int]] = None) -> str:
    filename = get_filename_from_size(mode, target_size)
    return os.path.join(RESIZED_PATH, f"{filename}.png")

def resize_map(mode: Union[PainMode, str], target_size: Optional[Tuple[int, int]] = None) -> bool:
    output_path = get_resized_map_path(mode, target_size)
    if target_size is None:
        if isinstance(mode, str):
            target_size = MAP_SIZE_DEFAULT
        elif mode in [PainMode.Physical, PainMode.Emotional, PainMode.SocioEcological]:
            target_size = MAP_SIZE_COUNTRIES
        elif mode == PainMode.Environmental:
            target_size = MAP_SIZE_CRATERS
        else:
            target_size = MAP_SIZE_SOUNDS
    img = Image.open(get_map_path(mode))
    img_new = img.resize(target_size)
    img_new.save(output_path)


In [19]:
for mode in PainMode:
    resize_map(mode)

In [50]:
# Sound Map for Fire, Earth, Water
img_fire = Image.open(get_resized_map_path(PainMode.Fire))
img_earth = Image.open(get_resized_map_path(PainMode.Earth))
img_water = Image.open(get_resized_map_path(PainMode.Water))

shape = img_fire.size
if img_earth.size != shape:
    raise Exception(f"Dimensions must be equal: {img_earth.size} != {shape}")
if img_water.size != shape:
    raise Exception(f"Dimensions must be equal: {img_water.size} != {shape}")
print(shape)

composite = np.ndarray(shape=(shape[1], shape[0], 4), dtype=np.uint8)
composite.fill(0)
for y in range(composite.shape[0]):
    for x in range(composite.shape[1]):
        r = img_fire.getpixel((x, y))[0]
        g = img_earth.getpixel((x, y))[1]
        b = img_water.getpixel((x, y))[2]
        if r != 0:
            debug_me = True
        composite[y, x] = (r, g, b, 255)

img_composite = Image.fromarray(composite, "RGBA")
img_composite.save(get_resized_map_path("comp_fire-earth-water"))

(600, 300)


C:\Users\Mike\AppData\Local\Temp\ipykernel_18472\2184013113.py:24: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img_composite = Image.fromarray(composite, "RGBA")


In [51]:
# Sound Map for Metal and Wood
img_metal = Image.open(get_resized_map_path(PainMode.Metal))
img_wood = Image.open(get_resized_map_path(PainMode.Wood))

shape = img_metal.size
if img_wood.size != shape:
    raise Exception(f"Dimensions must be equal: {img_wood.size} != {shape}")

composite = np.ndarray(shape=(shape[1], shape[0], 4), dtype=np.uint8)
composite.fill(0)
for y in range(composite.shape[0]):
    for x in range(composite.shape[1]):
        r = img_metal.getpixel((x, y))[0]
        g = img_wood.getpixel((x, y))[1]
        composite[y, x] = (r, g, 0, 255)

img_composite = Image.fromarray(composite, "RGBA")
img_composite.save(get_resized_map_path("comp_metal-wood"))

C:\Users\Mike\AppData\Local\Temp\ipykernel_18472\3496975578.py:17: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img_composite = Image.fromarray(composite, "RGBA")
